In [1]:
import tensorflow as tf
import os

### Configuracion para uso

In [3]:
DATA_DIR = "../../datasets/raw/images/breast"

# Definir caracteristicas de las imagenes
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
SEED = 42

In [ ]:
# ==============================
# 2. CREACIÓN DE DATASETS (train / val)
# ==============================

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",           # usa nombres de carpetas (benign, malignant)
    label_mode="binary",         # 0 o 1
    validation_split=0.2,        # 80% train, 20% val
    subset="training",
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="binary",
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print("Clases detectadas:", class_names)

Found 7909 files belonging to 2 classes.
Using 6328 files for training.
Found 7909 files belonging to 2 classes.
Using 1581 files for validation.
Clases detectadas: ['benign', 'malignant']


In [5]:
# ==============================
# 3. OPTIMIZAR PIPELINE (cache, prefetch)
# ==============================

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    train_ds
    .cache()
    .shuffle(1000)
    .prefetch(buffer_size=AUTOTUNE)
)

val_ds = (
    val_ds
    .cache()
    .prefetch(buffer_size=AUTOTUNE)
)

In [6]:
# ==============================
# 4. DEFINIR MODELO CNN
# ==============================

model = tf.keras.Sequential([
    # Normaliza píxeles [0,255] -> [0,1]
    tf.keras.layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),

    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')  # salida binaria: benigno/maligno
])

model.summary()

c:\Users\bruno\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# ==============================
# 5. COMPILAR MODELO
# ==============================

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [8]:
# ==============================
# 6. ENTRENAR MODELO
# ==============================

EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 135s 572ms/step - accuracy: 0.7468 - loss: 0.5887 - val_accuracy: 0.7944 - val_loss: 0.4803
Epoch 2/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 115s 582ms/step - accuracy: 0.8216 - loss: 0.4598 - val_accuracy: 0.8273 - val_loss: 0.4366
Epoch 3/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 113s 572ms/step - accuracy: 0.8382 - loss: 0.4242 - val_accuracy: 0.6863 - val_loss: 0.6640
Epoch 4/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 109s 552ms/step - accuracy: 0.8405 - loss: 0.4168 - val_accuracy: 0.8185 - val_loss: 0.4406
Epoch 5/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 119s 601ms/step - accuracy: 0.8500 - loss: 0.4084 - val_accuracy: 0.8286 - val_loss: 0.4117
Epoch 6/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 110s 556ms/step - accuracy: 0.8562 - loss: 0.3895 - val_accuracy: 0.8482 - val_loss: 0.3875
Epoch 7/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 109s 552ms/step - accuracy: 0.8541 - loss: 0.3820 - val_accuracy: 0.8571 - val_loss: 0.3846
Epoch 8/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 109s 551ms/step - accuracy: 0.8503 -

In [ ]:
# ==============================
# 7. GUARDAR MODELO
# ==============================

# Guarda en formato Keras estándar (carpeta con weights + config)
MODEL_DIR = "../results/cnn_breast_keras_10epochs.keras"
model.save(MODEL_DIR)
MODEL_DIR = "../results/cnn_breast_keras_10epochs.h5"
model.save(MODEL_DIR)

print(f"\nModelo guardado en: {MODEL_DIR}")


Modelo guardado en: ../results/cnn_breast_keras_10epochs.h5


### Volver a entrenar modelo pero con mas epocas

In [17]:
EPOCHS = 25

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/25
198/198 ━━━━━━━━━━━━━━━━━━━━ 108s 544ms/step - accuracy: 0.8671 - loss: 0.3608 - val_accuracy: 0.8533 - val_loss: 0.3843
Epoch 2/25
198/198 ━━━━━━━━━━━━━━━━━━━━ 101s 511ms/step - accuracy: 0.8759 - loss: 0.3405 - val_accuracy: 0.8590 - val_loss: 0.3784
Epoch 3/25
198/198 ━━━━━━━━━━━━━━━━━━━━ 107s 541ms/step - accuracy: 0.8706 - loss: 0.3446 - val_accuracy: 0.8514 - val_loss: 0.3835
Epoch 4/25
198/198 ━━━━━━━━━━━━━━━━━━━━ 107s 541ms/step - accuracy: 0.8712 - loss: 0.3333 - val_accuracy: 0.8438 - val_loss: 0.3785
Epoch 5/25
198/198 ━━━━━━━━━━━━━━━━━━━━ 109s 550ms/step - accuracy: 0.8744 - loss: 0.3274 - val_accuracy: 0.8545 - val_loss: 0.3722
Epoch 6/25
198/198 ━━━━━━━━━━━━━━━━━━━━ 108s 547ms/step - accuracy: 0.8823 - loss: 0.3091 - val_accuracy: 0.8583 - val_loss: 0.3759
Epoch 7/25
198/198 ━━━━━━━━━━━━━━━━━━━━ 109s 549ms/step - accuracy: 0.8946 - loss: 0.2814 - val_accuracy: 0.8653 - val_loss: 0.3529
Epoch 8/25
198/198 ━━━━━━━━━━━━━━━━━━━━ 107s 543ms/step - accuracy: 0.8908 -

In [18]:
MODEL_DIR = "../results/cnn_breast_keras_25epochs.keras"
model.save(MODEL_DIR)
MODEL_DIR = "../results/cnn_breast_keras_25epochs.h5"
model.save(MODEL_DIR)

print(f"\nModelo guardado en: {MODEL_DIR}")


Modelo guardado en: ../results/cnn_breast_keras_25epochs.h5
